# 📘 Notebook: CQED Molecular Dynamics with Frozen Atoms

🧠 Overview

In this notebook, we run velocity Verlet molecular dynamics using the CQED calculator.

We also show how to freeze one or more atoms during the trajectory by:

- zeroing the corresponding force components
- zeroing their velocities
- preventing their coordinates from updating

This is useful for anchoring part of a molecule while letting the other atoms evolve naturally.

In [1]:
import numpy as np
import psi4
import matplotlib.pyplot as plt

psi4.core.be_quiet()

from cqed_rhf import CQEDRHFCalculator
from cqed_rhf.utils import write_xyz, ANGSTROM_TO_BOHR

# 🧬 Example Geometry

Here we use a small example geometry of water with oxygen frozen. You can replace this with the nitrobenzene with Br+ anchored.

After defining the geometry, we will set some psi4 options, define the cavity vector, and instatiate the CQED calculator.

In [5]:
geometry = """
0 1
O   0.000000   0.000000  -0.124722
H   0.000000  -0.757000   0.494000
H   0.000000   0.757000   0.494000
units angstrom
no_reorient
no_com
symmetry c1
"""

psi4.set_memory("2 GB")
psi4.set_num_threads(2)

psi4_options = {
    "basis": "6-311G",
    "scf_type": "df",
    "e_convergence": 1e-10,
    "d_convergence": 1e-8,
}

lambda_vector = np.array([0.0, 0.0, 0.05])
omega = 0.07349864501573

calc = CQEDRHFCalculator(
    lambda_vector=lambda_vector,
    psi4_options=psi4_options,
    omega=omega,
    density_fitting=True,
    functional="wb97x-d", # set to "None" for QED-RHF
    charge=0,
    multiplicity=1,
    debug=False,
)

# 🧱 Build Initial Coordinates and Symbols

In [6]:
mol = psi4.geometry(geometry)
symbols = [mol.symbol(i) for i in range(mol.natom())]
coords_bohr = mol.geometry().to_array()
natom = mol.natom()

# ❄️ Choose Frozen Atoms

We define a Boolean mask of shape `(natom,).`

Example: freeze atom 0 (the oxygen).

In [7]:
freeze_atoms = np.zeros(natom, dtype=bool)
freeze_atoms[0] = True

print("Frozen atoms:", np.where(freeze_atoms)[0])

Frozen atoms: [0]
